# Fase 3: Visualización de clusters de emociones bajo compresión SVD

Visualizamos cómo los embeddings [CLS] se degradan progresivamente al comprimir el modelo
con SVD a distintos rangos. Usamos t-SNE sobre el conjunto concatenado de todos los modelos
para garantizar un espacio de coordenadas compartido.

**Contenido:**
1. Extracción de embeddings [CLS] del último hidden state
2. Proyección t-SNE conjunta (baseline + 4 niveles de compresión)
3. Panel comparativo de scatter plots coloreados por emoción dominante
4. Variante con contornos de densidad KDE
5. Zoom a emociones específicas — solapamiento progresivo

In [ ]:
import sys
import os

# En Colab, montar Drive y ajustar path al proyecto
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'

# En local
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from scipy.stats import gaussian_kde
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.data.dataset import NUM_LABELS, EMOTION_NAMES
from src.utils import compute_metrics
from src.compression import (
    apply_svd_compression,
    count_parameters,
    get_compression_ratio,
)

sns.set_style("whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Cargar modelo y datos

In [ ]:
model_path = os.path.join(PROJECT_ROOT, "results", "bert-goemotions-final")

baseline_model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)
baseline_model.eval()
baseline_model.to(device)

params = count_parameters(baseline_model)
print(f"Parámetros totales: {params['total']:,}")

In [ ]:
_, _, test_ds, emotion_names, data_collator = load_goemotions()
print(f"Test set: {len(test_ds)} ejemplos")

In [ ]:
# Helper de evaluación para obtener F1 macro
eval_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_ROOT, "results", "tmp_eval"),
    per_device_eval_batch_size=64,
    fp16=(device == "cuda"),
    report_to="none",
)

def evaluate_model(model):
    """Evalúa un modelo en el test set y devuelve el dict de métricas."""
    trainer = Trainer(
        model=model,
        args=eval_args,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    return trainer.evaluate(test_ds)

## 2. Función de extracción de embeddings [CLS]

In [ ]:
@torch.no_grad()
def extract_cls_embeddings(model, dataset, batch_size=64):
    """Extrae embeddings [CLS] del último hidden state.

    Hace forward pass con output_hidden_states=True y toma
    hidden_states[-1][:, 0, :] (token [CLS] de la última capa).

    Returns:
        embeddings: numpy array (N, hidden_dim)
        labels: numpy array (N, num_labels) multi-hot
    """
    model.eval()
    model_device = next(model.parameters()).device

    all_embeddings = []
    all_labels = []

    for i in range(0, len(dataset), batch_size):
        batch_indices = range(i, min(i + batch_size, len(dataset)))
        batch = data_collator([dataset[j] for j in batch_indices])

        input_ids = batch["input_ids"].to(model_device)
        attention_mask = batch["attention_mask"].to(model_device)
        labels = batch["labels"].numpy()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        # Último hidden state, token [CLS] (posición 0)
        cls_emb = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
        all_embeddings.append(cls_emb)
        all_labels.append(labels)

    return np.concatenate(all_embeddings, axis=0), np.concatenate(all_labels, axis=0)

## 3. Extraer embeddings — baseline + 4 niveles de compresión

In [ ]:
# Subsamplear test set para que t-SNE sea manejable
N_SAMPLES = 2000
rng = np.random.RandomState(42)
indices = rng.choice(len(test_ds), size=min(N_SAMPLES, len(test_ds)), replace=False)
indices.sort()
sub_test = test_ds.select(indices)
print(f"Subsample: {len(sub_test)} ejemplos")

In [ ]:
SVD_RANKS = [512, 256, 128, 64]
model_configs = [("Baseline", None)] + [(f"Rank {r}", r) for r in SVD_RANKS]

embeddings_dict = {}  # name -> (embeddings, labels)
f1_scores = {}  # name -> f1_macro

for name, rank in model_configs:
    print(f"\n{'='*50}")
    print(f"Procesando: {name}")

    if rank is None:
        model = baseline_model
    else:
        model = apply_svd_compression(baseline_model, rank=rank)
        model.to(device)

    # Evaluar F1 para los títulos
    metrics = evaluate_model(model)
    f1_scores[name] = metrics["eval_f1_macro"]
    print(f"  F1 macro: {metrics['eval_f1_macro']:.4f}")

    # Extraer embeddings
    embs, labels = extract_cls_embeddings(model, sub_test)
    embeddings_dict[name] = (embs, labels)
    print(f"  Embeddings shape: {embs.shape}")

    if rank is not None:
        ratio = get_compression_ratio(baseline_model, model)
        print(f"  Compresión: {ratio:.2f}x")
        del model
        if device == "cuda":
            torch.cuda.empty_cache()

print(f"\nModelos procesados: {list(embeddings_dict.keys())}")

## 4. Proyección t-SNE conjunta

In [ ]:
# Concatenar embeddings de TODOS los modelos
model_names = list(embeddings_dict.keys())
all_embs = np.concatenate([embeddings_dict[n][0] for n in model_names], axis=0)
n_per_model = len(sub_test)

print(f"Embeddings concatenados: {all_embs.shape}")
print(f"Puntos por modelo: {n_per_model}")
print(f"Total puntos para t-SNE: {all_embs.shape[0]}")

In [ ]:
# Ejecutar t-SNE una sola vez sobre el conjunto completo
print("Ejecutando t-SNE (puede tardar unos minutos)...")
tsne = TSNE(
    n_components=2,
    perplexity=30,
    n_iter=1000,
    random_state=42,
    init="pca",
    learning_rate="auto",
)
all_tsne = tsne.fit_transform(all_embs)
print(f"t-SNE completado. Shape: {all_tsne.shape}")

# Separar de vuelta por modelo
tsne_per_model = {}
for i, name in enumerate(model_names):
    start = i * n_per_model
    end = start + n_per_model
    tsne_per_model[name] = all_tsne[start:end]
    print(f"  {name}: {tsne_per_model[name].shape}")

## 5. Preparar coloreado por emoción dominante

In [ ]:
# Las labels son multi-hot — usar argmax como emoción dominante
labels = embeddings_dict[model_names[0]][1]  # mismas labels para todos
dominant_emotion = np.argmax(labels, axis=1)

# Encontrar las top 10 emociones más frecuentes en el subsample
unique, counts = np.unique(dominant_emotion, return_counts=True)
top_indices = unique[np.argsort(-counts)][:10]
top_emotion_names = [EMOTION_NAMES[i] for i in top_indices]

print("Top 10 emociones dominantes:")
for idx in top_indices:
    count = counts[unique == idx][0]
    print(f"  {EMOTION_NAMES[idx]:>15s}: {count:>4d} ({count/len(dominant_emotion)*100:.1f}%)")

# Asignar colores: top 10 con colores distintos, resto gris
cmap = plt.cm.tab10
color_map = {}
for rank_i, emo_idx in enumerate(top_indices):
    color_map[emo_idx] = cmap(rank_i)

colors = np.array([color_map.get(e, (0.8, 0.8, 0.8, 0.3)) for e in dominant_emotion])
# Máscara para saber cuáles son top emociones (para leyenda)
is_top = np.isin(dominant_emotion, top_indices)

## 6. Panel principal — scatter plots por modelo

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(30, 6))

# Ejes compartidos
x_min, x_max = all_tsne[:, 0].min() - 2, all_tsne[:, 0].max() + 2
y_min, y_max = all_tsne[:, 1].min() - 2, all_tsne[:, 1].max() + 2

for ax, name in zip(axes, model_names):
    coords = tsne_per_model[name]

    # Plotear puntos "otros" primero (gris, detrás)
    other_mask = ~is_top
    ax.scatter(
        coords[other_mask, 0], coords[other_mask, 1],
        c=[(0.85, 0.85, 0.85, 0.2)],
        s=3, rasterized=True,
    )

    # Plotear top emociones encima
    for rank_i, emo_idx in enumerate(top_indices):
        mask = dominant_emotion == emo_idx
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c=[cmap(rank_i)],
            s=8, alpha=0.6,
            label=EMOTION_NAMES[emo_idx] if ax == axes[0] else None,
            rasterized=True,
        )

    f1 = f1_scores[name]
    ax.set_title(f"{name}\nF1 macro = {f1:.4f}", fontsize=12, fontweight="bold")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks([])
    ax.set_yticks([])

# Leyenda compartida
axes[0].legend(
    bbox_to_anchor=(-0.15, 0.5), loc="center right",
    fontsize=9, frameon=True, framealpha=0.9,
    markerscale=3,
)

fig.suptitle(
    "Evolución de clusters de emociones bajo compresión SVD\n"
    "(t-SNE de embeddings [CLS], último hidden state)",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
os.makedirs(os.path.join(PROJECT_ROOT, "results"), exist_ok=True)
plt.savefig(
    os.path.join(PROJECT_ROOT, "results", "embedding_clusters_svd.png"),
    dpi=200, bbox_inches="tight",
)
plt.show()
print("Figura guardada en results/embedding_clusters_svd.png")

## 7. Variante con densidad KDE

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(30, 6))

# Seleccionar top 6 emociones para contornos KDE (más limpias)
top_for_kde = top_indices[:6]

for ax, name in zip(axes, model_names):
    coords = tsne_per_model[name]

    # Fondo: todos los puntos en gris muy claro
    ax.scatter(
        coords[:, 0], coords[:, 1],
        c=[(0.9, 0.9, 0.9, 0.15)],
        s=1, rasterized=True,
    )

    # Contornos KDE por emoción
    for rank_i, emo_idx in enumerate(top_for_kde):
        mask = dominant_emotion == emo_idx
        if mask.sum() < 10:
            continue

        x = coords[mask, 0]
        y = coords[mask, 1]

        try:
            kde = gaussian_kde(np.vstack([x, y]), bw_method=0.3)
            xi = np.linspace(x_min, x_max, 100)
            yi = np.linspace(y_min, y_max, 100)
            xi, yi = np.meshgrid(xi, yi)
            zi = kde(np.vstack([xi.ravel(), yi.ravel()])).reshape(xi.shape)

            color = cmap(rank_i)
            ax.contour(
                xi, yi, zi,
                levels=3, colors=[color],
                linewidths=1.5, alpha=0.7,
            )
            ax.contourf(
                xi, yi, zi,
                levels=3, colors=[(*color[:3], 0.05), (*color[:3], 0.1), (*color[:3], 0.15), (*color[:3], 0.2)],
            )
        except np.linalg.LinAlgError:
            # KDE puede fallar si los puntos son colineales
            pass

    f1 = f1_scores[name]
    ax.set_title(f"{name}\nF1 macro = {f1:.4f}", fontsize=12, fontweight="bold")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks([])
    ax.set_yticks([])

# Leyenda manual para KDE
from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=cmap(i), alpha=0.5, label=EMOTION_NAMES[idx])
    for i, idx in enumerate(top_for_kde)
]
axes[0].legend(
    handles=legend_handles,
    bbox_to_anchor=(-0.15, 0.5), loc="center right",
    fontsize=9, frameon=True, framealpha=0.9,
)

fig.suptitle(
    "Densidad de emociones bajo compresión SVD\n"
    "(contornos KDE sobre t-SNE de embeddings [CLS])",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig(
    os.path.join(PROJECT_ROOT, "results", "embedding_density_svd.png"),
    dpi=200, bbox_inches="tight",
)
plt.show()
print("Figura guardada en results/embedding_density_svd.png")

## 8. Zoom a emociones específicas — solapamiento progresivo

In [ ]:
# Seleccionar 4 emociones interesantes para el análisis de solapamiento
# Buscamos emociones que sean semánticamente cercanas y probablemente se solapen
zoom_emotions = ["joy", "love", "sadness", "anger"]
zoom_indices = [EMOTION_NAMES.index(e) for e in zoom_emotions]
zoom_colors = ["#2ecc71", "#e74c3c", "#3498db", "#e67e22"]

print("Emociones para zoom:")
for e, idx in zip(zoom_emotions, zoom_indices):
    count = (dominant_emotion == idx).sum()
    print(f"  {e}: {count} ejemplos")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(30, 6))

for ax, name in zip(axes, model_names):
    coords = tsne_per_model[name]

    # Fondo gris
    ax.scatter(
        coords[:, 0], coords[:, 1],
        c=[(0.92, 0.92, 0.92, 0.1)],
        s=1, rasterized=True,
    )

    # Scatter + contornos para cada emoción seleccionada
    for emo_name, emo_idx, color in zip(zoom_emotions, zoom_indices, zoom_colors):
        mask = dominant_emotion == emo_idx
        if mask.sum() < 5:
            continue

        x = coords[mask, 0]
        y = coords[mask, 1]

        # Scatter
        ax.scatter(
            x, y, c=[color], s=12, alpha=0.5,
            label=emo_name if ax == axes[0] else None,
            rasterized=True, zorder=2,
        )

        # Contorno KDE
        if mask.sum() >= 15:
            try:
                kde = gaussian_kde(np.vstack([x, y]), bw_method=0.3)
                xi = np.linspace(x_min, x_max, 100)
                yi = np.linspace(y_min, y_max, 100)
                xi, yi = np.meshgrid(xi, yi)
                zi = kde(np.vstack([xi.ravel(), yi.ravel()])).reshape(xi.shape)

                ax.contour(
                    xi, yi, zi,
                    levels=2, colors=[color],
                    linewidths=2, alpha=0.8, zorder=3,
                )
            except np.linalg.LinAlgError:
                pass

    f1 = f1_scores[name]
    ax.set_title(f"{name}\nF1 macro = {f1:.4f}", fontsize=12, fontweight="bold")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks([])
    ax.set_yticks([])

axes[0].legend(
    bbox_to_anchor=(-0.15, 0.5), loc="center right",
    fontsize=11, frameon=True, framealpha=0.9,
    markerscale=3,
)

fig.suptitle(
    "Solapamiento progresivo de emociones bajo compresión SVD\n"
    f"({', '.join(zoom_emotions)})",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig(
    os.path.join(PROJECT_ROOT, "results", "embedding_zoom_emotions_svd.png"),
    dpi=200, bbox_inches="tight",
)
plt.show()
print("Figura guardada en results/embedding_zoom_emotions_svd.png")

## 9. Métricas cuantitativas de separación de clusters

In [ ]:
from sklearn.metrics import silhouette_score

# Silhouette score usando solo las top emociones (filtrando "otros")
top_mask = is_top
top_labels = dominant_emotion[top_mask]

print(f"{'Modelo':<15s} | {'Silhouette':>10s} | {'F1 macro':>9s}")
print("-" * 42)

for name in model_names:
    coords = tsne_per_model[name][top_mask]
    sil = silhouette_score(coords, top_labels, sample_size=min(1000, len(coords)))
    f1 = f1_scores[name]
    print(f"{name:<15s} | {sil:>10.4f} | {f1:>9.4f}")

## 10. Resumen

In [ ]:
print("=" * 60)
print("RESUMEN — Fase 3: Visualización de embeddings")
print("=" * 60)
print(f"\nEjemplos visualizados: {N_SAMPLES}")
print(f"Modelos comparados: {len(model_names)}")
print(f"Emociones coloreadas: {len(top_indices)} (top por frecuencia)")
print(f"\nF1 macro por modelo:")
for name in model_names:
    print(f"  {name:<15s}: {f1_scores[name]:.4f}")
print(f"\nFiguras guardadas en results/:")
print(f"  - embedding_clusters_svd.png")
print(f"  - embedding_density_svd.png")
print(f"  - embedding_zoom_emotions_svd.png")